# Train Nutrition Tracker's food models on free Google Colab

This notebook trains the app's two photo models and exports them for Core ML:

- **Model A (recognition):** works out which foods are in a photo.
- **Model B (portion and nutrition):** estimates how much is there.

**Before you start**
1. **Runtime → Change runtime type → T4 GPU.**
2. You need about **5 GB free in Google Drive**. Checkpoints and cached datasets are saved there, because Colab wipes its own disk when the session ends.
3. Push this repo to GitHub, or set `REPO_URL` below to wherever it lives.

**Free Colab will disconnect you.** Sessions end after a few hours, or sooner if the tab sits idle. That's expected:
- Each training run saves after every epoch and stops cleanly before `SESSION_MINUTES` runs out.
- To continue, reconnect, then **Runtime → Run all**. Training picks up from the last saved epoch, and datasets restore from Drive in a minute or two instead of downloading again.

**Rough timings.** These are my estimates, not measurements; watch the per-epoch time it prints. Free Colab gives 2 CPU cores, so loading images, not the GPU, is usually the bottleneck.
- **Model A (40 epochs on FoodSeg103):** probably several hours, spread over 2–4 sessions.
- **Model B (60 epochs on Nutrition5k):** probably a few hours, spread over 1–3 sessions.

You can run the two sections in separate sessions, in either order.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/FelixYSW/Nutrition-Tracker.git"
BRANCH = "main"
# Only needed if the repo is private: a fine-grained, read-only GitHub token.
GITHUB_TOKEN = ""

DRIVE_DIR = "/content/drive/MyDrive/NutritionTracker"

# FoodSeg103 (Model A). Download the zip yourself from
# https://xiongweiwu.github.io/foodseg103.html (it asks you to accept research-use
# terms and gives the password), then upload it to this Drive path:
FOODSEG_ZIP = f"{DRIVE_DIR}/datasets/FoodSeg103.zip"
FOODSEG_PASSWORD = ""

# Stop each training run with a safety margin before Colab cuts the session.
SESSION_MINUTES = 150

# Free Colab has 2 CPU cores.
WORKERS = 2

## 2. Check the GPU, mount Drive, fetch the code

In [ ]:
import os, subprocess, torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then rerun."
print("GPU:", torch.cuda.get_device_name(0))
tv = tuple(int(p) for p in torchvision.__version__.split("+")[0].split(".")[:2])
assert tv >= (0, 17), "torchvision 0.17+ needed for the training transforms."

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
for sub in ("datasets", "runs", "exports"):
    os.makedirs(f"{DRIVE_DIR}/{sub}", exist_ok=True)

In [ ]:
REPO = "/content/Nutrition-Tracker"
url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
if os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    # Run via subprocess and never echo the URL, so a token can't leak into the output.
    result = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise SystemExit("Clone failed. Check REPO_URL / BRANCH, and GITHUB_TOKEN if the repo is private.")
os.chdir(REPO)
print("Code ready at", REPO)

In [ ]:
# Colab already has torch, torchvision, numpy and pillow. Only coremltools is added,
# so nothing preinstalled gets downgraded.
!pip install -q "coremltools>=8.0" tqdm

DATA = "/content/data"                 # fast local disk, rebuilt each session
os.environ["NT_DATA_DIR"] = DATA
os.environ["NT_RUNS_DIR"] = f"{DRIVE_DIR}/runs"   # checkpoints survive disconnects
os.makedirs(f"{DATA}/raw", exist_ok=True)
print("Checkpoints will be saved to", os.environ["NT_RUNS_DIR"])

## 3. Model A: food recognition (FoodSeg103)

Needs `FOODSEG_ZIP` and `FOODSEG_PASSWORD` set in step 1.

In [ ]:
import glob
foodseg_root = None
found = glob.glob(f"{DATA}/raw/**/category_id.txt", recursive=True)
if found:
    foodseg_root = os.path.dirname(found[0])
elif os.path.exists(FOODSEG_ZIP):
    print("Extracting FoodSeg103 from Drive...")
    pw = ["-P", FOODSEG_PASSWORD] if FOODSEG_PASSWORD else []
    subprocess.run(["unzip", "-q", "-o", *pw, FOODSEG_ZIP, "-d", f"{DATA}/raw"], check=True)
    found = glob.glob(f"{DATA}/raw/**/category_id.txt", recursive=True)
    if not found:
        raise SystemExit("Extracted, but no category_id.txt was found. Is this the FoodSeg103 archive?")
    foodseg_root = os.path.dirname(found[0])
else:
    raise SystemExit(f"Upload the FoodSeg103 zip to {FOODSEG_ZIP} first (see step 1).")

print("FoodSeg103 at", foodseg_root)
!python -m ml.scripts.prepare_foodseg103 --root "{foodseg_root}"

In [ ]:
# Train. Safe to rerun after a disconnect: --resume continues from the last epoch.
!python -m ml.ingredient_segmentation.train --stage base \
    --manifests {DATA}/manifests/foodseg103.jsonl \
    --batch-size 16 --workers {WORKERS} \
    --resume --max-minutes {SESSION_MINUTES}

In [ ]:
# Check progress. Rerun after every session; "epoch" shows where training is up to.
last = f"{DRIVE_DIR}/runs/model_a_base/last.pt"
if os.path.exists(last):
    state = torch.load(last, map_location="cpu", weights_only=False)
    for row in state["history"][-5:]:
        print({k: row[k] for k in ("epoch", "seconds", "train_loss", "miou") if k in row})
    print(f"Completed {state['epoch']} of 40 epochs.")
else:
    print("No checkpoint yet.")

In [ ]:
# Full evaluation, once training has finished.
!python -m ml.ingredient_segmentation.evaluate \
    --checkpoint "{DRIVE_DIR}/runs/model_a_base/best.pt" \
    --manifests {DATA}/manifests/foodseg103.jsonl --split val

## 4. Model B: portion and nutrition (Nutrition5k)

On the first run this downloads the overhead RGB photos and metadata from Google's public Nutrition5k bucket. Depth images are skipped, because the app can't use them yet. The result is cached as one archive in Drive, so later sessions just unpack it.

I couldn't check the download size from here, so the cell prints it. If it's too big for your Drive, delete the archive afterwards; you'll just re-download each session.

In [ ]:
N5K = f"{DATA}/raw/nutrition5k_dataset"
N5K_CACHE = f"{DRIVE_DIR}/datasets/nutrition5k_overhead_rgb.tar"
BUCKET = "gs://nutrition5k_dataset/nutrition5k_dataset"

if os.path.isdir(f"{N5K}/metadata"):
    print("Nutrition5k already on local disk.")
elif os.path.exists(N5K_CACHE):
    print("Restoring Nutrition5k from Drive cache...")
    subprocess.run(["tar", "-xf", N5K_CACHE, "-C", f"{DATA}/raw"], check=True)
else:
    os.makedirs(f"{N5K}/imagery/realsense_overhead", exist_ok=True)
    !gsutil -m -q cp -r {BUCKET}/metadata {N5K}/
    !gsutil -m -q cp -r {BUCKET}/dish_ids {N5K}/
    # Excludes the depth images: only rgb.png per dish is needed for the RGB model.
    !gsutil -m -q rsync -r -x ".*depth.*" {BUCKET}/imagery/realsense_overhead {N5K}/imagery/realsense_overhead
    !du -sh {N5K}
    print("Caching to Drive for next session...")
    subprocess.run(["tar", "-cf", N5K_CACHE, "-C", f"{DATA}/raw", "nutrition5k_dataset"], check=True)

!python -m ml.scripts.prepare_nutrition5k --root {N5K}

In [ ]:
# Train the RGB model, the only variant the app can use in V1. Safe to rerun after a disconnect.
!python -m ml.nutrition_estimation.train --inputs rgb \
    --batch-size 32 --workers {WORKERS} \
    --resume --max-minutes {SESSION_MINUTES}

In [ ]:
# Final test-set evaluation (mean absolute error per target), once training has finished.
!python -m ml.nutrition_estimation.evaluate \
    --checkpoint "{DRIVE_DIR}/runs/model_b_rgb_mobilenet_v3_large/best.pt" --split test

## 5. Export to Core ML

This produces `IngredientSegmenter.mlpackage` and `NutritionEstimator.mlpackage`, zips them into Drive, and downloads them to your computer.

Core ML *conversion* works on Colab's Linux machines. Core ML *prediction* only runs on macOS, so the export script skips its self-test here; that's expected.

In [ ]:
import shutil
from google.colab import files

MODEL_A_CHECKPOINT = f"{DRIVE_DIR}/runs/model_a_base/best.pt"   # or model_a_malaysian (section 6)
exports = {
    "IngredientSegmenter": ("ml.ingredient_segmentation.export", MODEL_A_CHECKPOINT),
    "NutritionEstimator": ("ml.nutrition_estimation.export",
                           f"{DRIVE_DIR}/runs/model_b_rgb_mobilenet_v3_large/best.pt"),
}
shutil.rmtree("/content/exports", ignore_errors=True)
os.makedirs("/content/exports")
exported = []
for name, (module, checkpoint) in exports.items():
    if not os.path.exists(checkpoint):
        print(f"Skipping {name}: no checkpoint yet at {checkpoint}")
        continue
    out = f"/content/exports/{name}.mlpackage"
    subprocess.run(["python", "-m", module, "--checkpoint", checkpoint, "--out", out], check=True)
    exported.append(name)

if exported:
    # One zip holding both .mlpackage folders at the top level: the layout the
    # GitHub Actions "Fetch trained models" step expects.
    archive = shutil.make_archive(f"{DRIVE_DIR}/exports/NutritionTrackerModels", "zip", "/content/exports")
    print("Saved", archive, "containing", exported)
    print(f"{os.path.getsize(archive) / 1e6:.1f} MB")
    files.download(archive)

### Get the models into the app

**Through GitHub Actions** (builds go straight to your phone via Sideloadly):
1. In Google Drive, find `NutritionTracker/exports/NutritionTrackerModels.zip`, then choose **Share → General access → Anyone with the link → Copy link**.
2. On GitHub: repo → **Settings → Secrets and variables → Actions → New repository secret**. Name it `MODELS_URL` and paste the link as the value.
3. Run the workflow. Its log should say "Bundling IngredientSegmenter.mlpackage". In the app, Settings → AI shows both models as **Installed**.

When you retrain, rerun this export. It overwrites the same Drive file, and the link stays the same, so you don't need to update the secret.

**Privacy note:** the models never go into your public repo. However, the IPA artifact contains them, and on a public repo any signed-in GitHub user can download artifacts. For a personal app that's a small risk, but be aware of it.

**On a Mac instead:** unzip into `NutritionTracker/Resources/` and run `xcodegen generate`.

## 6. Optional: Malaysian fine-tuning (Model A, stage 2)

This teaches Model A local dishes using the Roboflow *Malaysian Food Recognition* sets (CC BY 4.0). It needs:
- a free Roboflow API key
- each dataset's `workspace/project/version`, copied from its Roboflow Universe page

Expect only modest gains: these sets are small. Finish section 3 first, then come back and set `MODEL_A_CHECKPOINT` in step 5 to `model_a_malaysian`.

In [ ]:
ROBOFLOW_API_KEY = ""
MFR1_PROJECT = ""   # e.g. "workspace/project/1", from the Roboflow Universe page
MFR2_PROJECT = ""

if not (ROBOFLOW_API_KEY and MFR1_PROJECT):
    raise SystemExit("Fill in ROBOFLOW_API_KEY and at least MFR1_PROJECT to run this section.")

!pip install -q roboflow
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
manifests = [f"{DATA}/manifests/foodseg103.jsonl"]
for project, folder in ((MFR1_PROJECT, "mfr1"), (MFR2_PROJECT, "mfr2")):
    if not project:
        continue
    ws, proj, ver = project.split("/")
    rf.workspace(ws).project(proj).version(int(ver)).download("coco", location=f"{DATA}/raw/{folder}")
    name = f"roboflow_{folder}"
    !python -m ml.scripts.prepare_malaysian roboflow --root {DATA}/raw/{folder} --name {name}
    manifests.append(f"{DATA}/manifests/{name}.jsonl")
MANIFEST_ARGS = " ".join(manifests)
print(MANIFEST_ARGS)

In [ ]:
!python -m ml.ingredient_segmentation.train --stage malaysian \
    --init "{DRIVE_DIR}/runs/model_a_base/best.pt" \
    --manifests {MANIFEST_ARGS} --source-weight foodseg103=0.3 \
    --batch-size 16 --workers {WORKERS} \
    --resume --max-minutes {SESSION_MINUTES}